In [6]:
import pandas as pd
import os

# Definir la ruta relativa desde la carpeta 'scr' hacia 'data'
csv_path = os.path.join("..", "data", "Evolución_enfermedad_fiebre_de_Norilsk.csv")

# Cargar el CSV
df = pd.read_csv(csv_path, header=[0, 1], index_col=0)

# Mostrar el DataFrame
print(df)


            Semana 1 Unnamed: 2_level_0 Unnamed: 3_level_0    Semana 2  \
         Susceptible         Infectados            Muertos Susceptible   
Norilsk       200000                300                  0      200000   
Kayerkan       80000                 10                  0       80000   
Dudinka        60000                 20                  0       60000   

         Unnamed: 5_level_0 Unnamed: 6_level_0    Semana 3 Unnamed: 8_level_0  \
                 Infectados            Muertos Susceptible         Infectados   
Norilsk                1500                300      200000               5000   
Kayerkan                700                120       80000               2300   
Dudinka                1200                 60       60000               3000   

         Unnamed: 9_level_0  
                    Muertos  
Norilsk                 650  
Kayerkan               1000  
Dudinka                 900  


In [9]:
df

,Semana 1_Susceptible,Semana 1_Infectados,Semana 1_Muertos,Semana 2_Susceptible,Semana 2_Infectados,Semana 2_Muertos,Semana 3_Susceptible,Semana 3_Infectados,Semana 3_Muertos
Norilsk,200000,300,0,200000,1500,300,200000,5000,650
Kayerkan,80000,10,0,80000,700,120,80000,2300,1000
Dudinka,60000,20,0,60000,1200,60,60000,3000,900


In [7]:
df.columns = [
    "Semana 1_Susceptible", "Semana 1_Infectados", "Semana 1_Muertos",
    "Semana 2_Susceptible", "Semana 2_Infectados", "Semana 2_Muertos",
    "Semana 3_Susceptible", "Semana 3_Infectados", "Semana 3_Muertos"
]

# Función para calcular R0
def calcular_r0(df):
    r0_resultados = {}

    for i in range(1, 3):  # Comparar Semana 1 -> Semana 2 y Semana 2 -> Semana 3
        S_t = df[f'Semana {i}_Susceptible']
        I_t = df[f'Semana {i}_Infectados']
        S_t1 = df[f'Semana {i+1}_Susceptible']
        I_t1 = df[f'Semana {i+1}_Infectados']

        # Estimación de β (tasa de infección) y γ (tasa de recuperación/muerte)
        beta = (I_t1 - I_t) / (S_t * I_t)
        gamma = (I_t - I_t1) / I_t

        # Evitar divisiones por cero
        gamma.replace(0, 1e-6, inplace=True)

        # Cálculo de R0
        R0 = beta / gamma
        r0_resultados[f"Semana {i} -> Semana {i+1}"] = R0

    return pd.DataFrame(r0_resultados, index=df.index)

# Calcular R0
df_r0 = calcular_r0(df)
print(df_r0)


          Semana 1 -> Semana 2  Semana 2 -> Semana 3
Norilsk              -0.000005             -0.000005
Kayerkan             -0.000013             -0.000013
Dudinka              -0.000017             -0.000017


In [8]:
df_r0

,Semana 1 -> Semana 2,Semana 2 -> Semana 3
Norilsk,-0.000005,-0.000005
Kayerkan,-0.000013,-0.000013
Dudinka,-0.000017,-0.000017


In [5]:
print(df.shape)  # Esto imprimirá (número de filas, número de columnas)
print(df.columns)  # Esto mostrará los nombres actuales de las columnas


(3, 9)
MultiIndex([(          'Semana 1', 'Susceptible'),
            ('Unnamed: 2_level_0',  'Infectados'),
            ('Unnamed: 3_level_0',     'Muertos'),
            (          'Semana 2', 'Susceptible'),
            ('Unnamed: 5_level_0',  'Infectados'),
            ('Unnamed: 6_level_0',     'Muertos'),
            (          'Semana 3', 'Susceptible'),
            ('Unnamed: 8_level_0',  'Infectados'),
            ('Unnamed: 9_level_0',     'Muertos')],
           )


In [10]:
# Calcular R0
r0_values = []

for city in df.index:
    # Extraer los valores de infectados y susceptibles por semana
    susceptible_1 = df.loc[city, 'Semana 1_Susceptible']
    susceptible_2 = df.loc[city, 'Semana 2_Susceptible']
    susceptible_3 = df.loc[city, 'Semana 3_Susceptible']
    
    infectados_1 = df.loc[city, 'Semana 1_Infectados']
    infectados_2 = df.loc[city, 'Semana 2_Infectados']
    infectados_3 = df.loc[city, 'Semana 3_Infectados']
    
    # Calcular el R0 para cada semana
    r0_2 = (infectados_2 - infectados_1) / (susceptible_1 - susceptible_2)
    r0_3 = (infectados_3 - infectados_2) / (susceptible_2 - susceptible_3)
    
    # Promediar los valores de R0
    r0_avg = (r0_2 + r0_3) / 2
    r0_values.append(r0_avg)

# Asignar los valores de R0 al DataFrame
df_r0 = pd.DataFrame(r0_values, index=df.index, columns=['R0'])
print(df_r0)

           R0
Norilsk   inf
Kayerkan  inf
Dudinka   inf


C:\Users\felip\AppData\Local\Temp\ipykernel_27860\4264637498.py:15: RuntimeWarning: divide by zero encountered in scalar divide
  r0_2 = (infectados_2 - infectados_1) / (susceptible_1 - susceptible_2)
C:\Users\felip\AppData\Local\Temp\ipykernel_27860\4264637498.py:16: RuntimeWarning: divide by zero encountered in scalar divide
  r0_3 = (infectados_3 - infectados_2) / (susceptible_2 - susceptible_3)
